# Experiment 5: Binary Classification using Decision Trees and Random Forests

---
## Objective
The objective of this experiment is to construct, optimize, and compare Decision Tree and Random Forest classifiers for predicting breast cancer malignancy. 

In [ ]:
# Global Imports & Plot Styling Setup (Rules.md Compliance)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

# Enforce rules.md Plot Formatting Guidelines
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['font.size'] = 15
plt.rcParams['axes.labelsize'] = 15
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 15
plt.rcParams['xtick.labelsize'] = 15
plt.rcParams['ytick.labelsize'] = 15
plt.rcParams['legend.fontsize'] = 15
plt.rcParams['figure.titlesize'] = 16

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported and global Times New Roman (15pt, Bold Labels) plot formatting applied.")


In [ ]:
# Load Breast Cancer Wisconsin Dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='Diagnosis')

# Map target: 0 -> Malignant (M), 1 -> Benign (B)
df = pd.concat([X, y], axis=1)

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns ({X.shape[1]} features + 1 target)")
print(f"Target Distribution:\n{df['Diagnosis'].value_counts()}")
df.head()


In [ ]:
# Reusable Function 4.1: Consolidated 12-Subplot EDA (Rules.md Compliance)
def plot_12_subplots_eda(df, target_col='Diagnosis', save_prefix='exp5_eda_12_subplots'):
    """
    Generates a single consolidated figure containing 12 distinct EDA subplots on one page,
    conforming strictly to rules.md specifications (Times New Roman, 15pt font, bold labels).
    Exports the resulting figure in .webp format and EPS format.
    """
    fig, axes = plt.subplots(3, 4, figsize=(22, 16))
    plt.subplots_adjust(wspace=0.35, hspace=0.45)
    
    # 1. Target Class Distribution
    counts = df[target_col].value_counts()
    axes[0, 0].bar(['Benign (1)', 'Malignant (0)'], counts.values, color=['#2ca02c', '#d62728'], edgecolor='black')
    axes[0, 0].set_title('1. Target Class Distribution')
    axes[0, 0].set_xlabel('Diagnosis Class')
    axes[0, 0].set_ylabel('Count')
    
    # 2. Missing Value Analysis
    missing = df.isnull().sum()
    axes[0, 1].bar(range(len(missing)), missing.values, color='#1f77b4', edgecolor='black')
    axes[0, 1].set_title('2. Missing Value Count')
    axes[0, 1].set_xlabel('Feature Index')
    axes[0, 1].set_ylabel('Missing Count')
    axes[0, 1].set_ylim(0, 10)
    
    # 3. Duplicate Rows Pie Chart
    duplicates = df.duplicated().sum()
    axes[0, 2].pie([len(df) - duplicates, max(duplicates, 1)], labels=['Unique', 'Duplicate'],
                   autopct='%1.1f%%', colors=['#4c72b0', '#c44e52'])
    axes[0, 2].set_title('3. Duplicate Row Distribution')
    
    # 4. Top Feature Correlation Heatmap
    num_df = df.select_dtypes(include=np.number)
    corr = num_df.iloc[:, :8].corr()
    sns.heatmap(corr, ax=axes[0, 3], cmap='coolwarm', cbar=False, annot=False, rasterized=True)
    axes[0, 3].set_title('4. Correlation Heatmap (8 Feat)')
    axes[0, 3].set_xlabel('Features')
    axes[0, 3].set_ylabel('Features')
    
    # 5. Histogram - Mean Radius
    sns.histplot(data=df, x='mean radius', hue=target_col, kde=True, ax=axes[1, 0], palette=['#d62728', '#2ca02c'])
    axes[1, 0].set_title('5. Mean Radius Distribution')
    axes[1, 0].set_xlabel('Mean Radius')
    axes[1, 0].set_ylabel('Frequency')
    
    # 6. Histogram - Mean Texture
    sns.histplot(data=df, x='mean texture', hue=target_col, kde=True, ax=axes[1, 1], palette=['#d62728', '#2ca02c'])
    axes[1, 1].set_title('6. Mean Texture Distribution')
    axes[1, 1].set_xlabel('Mean Texture')
    axes[1, 1].set_ylabel('Frequency')
    
    # 7. Boxplot - Mean Perimeter
    sns.boxplot(data=df, y='mean perimeter', x=target_col, ax=axes[1, 2], palette=['#d62728', '#2ca02c'])
    axes[1, 2].set_title('7. Mean Perimeter Outliers')
    axes[1, 2].set_xlabel('Diagnosis Class')
    axes[1, 2].set_ylabel('Mean Perimeter')
    
    # 8. Boxplot - Mean Area
    sns.boxplot(data=df, y='mean area', x=target_col, ax=axes[1, 3], palette=['#d62728', '#2ca02c'])
    axes[1, 3].set_title('8. Mean Area Outliers')
    axes[1, 3].set_xlabel('Diagnosis Class')
    axes[1, 3].set_ylabel('Mean Area')
    
    # 9. Scatter Plot - Concavity vs Concave Points
    sns.scatterplot(data=df, x='mean concavity', y='mean concave points', hue=target_col, ax=axes[2, 0], palette=['#d62728', '#2ca02c'])
    axes[2, 0].set_title('9. Concavity vs Concave Pts')
    axes[2, 0].set_xlabel('Mean Concavity')
    axes[2, 0].set_ylabel('Mean Concave Pts')
    
    # 10. Histogram - Mean Smoothness
    sns.histplot(data=df, x='mean smoothness', hue=target_col, kde=True, ax=axes[2, 1], palette=['#d62728', '#2ca02c'])
    axes[2, 1].set_title('10. Mean Smoothness Dist')
    axes[2, 1].set_xlabel('Mean Smoothness')
    axes[2, 1].set_ylabel('Frequency')
    
    # 11. Boxplot - Worst Radius
    sns.boxplot(data=df, y='worst radius', x=target_col, ax=axes[2, 2], palette=['#d62728', '#2ca02c'])
    axes[2, 2].set_title('11. Worst Radius Outliers')
    axes[2, 2].set_xlabel('Diagnosis Class')
    axes[2, 2].set_ylabel('Worst Radius')
    
    # 12. Feature Distribution - Worst Area
    sns.histplot(data=df, x='worst area', hue=target_col, kde=True, ax=axes[2, 3], palette=['#d62728', '#2ca02c'])
    axes[2, 3].set_title('12. Worst Area Distribution')
    axes[2, 3].set_xlabel('Worst Area')
    axes[2, 3].set_ylabel('Frequency')
    
    # Export figures
    plt.tight_layout()
    webp_path = f"{save_prefix}.webp"
    eps_path = f"{save_prefix}.eps"
    
    fig.savefig(webp_path, format='webp', dpi=300, bbox_inches='tight')
    fig.savefig(eps_path, format='eps', dpi=100, bbox_inches='tight')
    plt.show()
    print(f"Consolidated 12-Subplot EDA saved as '{webp_path}' and '{eps_path}'.")

# Run 12-Subplot EDA function
plot_12_subplots_eda(df, save_prefix='exp5_eda_12_subplots')


In [ ]:
# Data Preprocessing & Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Standardize Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training Set: {X_train.shape[0]} samples")
print(f"Testing Set : {X_test.shape[0]} samples")


In [ ]:
# Reusable Function 4.3 & 4.5: Model Trainer & Classification Metrics Calculator (Rules.md Compliance)
def train_and_evaluate_classification_model(estimator, param_grid, X_train, y_train, cv=5):
    """
    Reusable function to train and tune classification models using GridSearchCV with Stratified K-Fold.
    """
    grid_search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=StratifiedKFold(n_splits=cv, shuffle=True, random_state=42),
        scoring='f1',
        n_jobs=-1
    )
    grid_search.fit(X_train, y_train)
    return grid_search.best_estimator_, grid_search.best_params_, grid_search.best_score_

def compute_classification_metrics(y_true, y_pred, y_prob):
    """
    Reusable function to compute and display all standard classification performance metrics.
    """
    metrics = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, pos_label=0), # Malignant positive class
        'Recall': recall_score(y_true, y_pred, pos_label=0),
        'F1-Score': f1_score(y_true, y_pred, pos_label=0),
        'ROC-AUC': roc_auc_score(y_true, 1 - y_prob) if y_prob is not None else np.nan
    }
    return metrics


In [ ]:
# 1. Train and Tune Decision Tree Classifier
dt_base = DecisionTreeClassifier(random_state=42)

param_grid_dt = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4, 8]
}

best_dt, dt_params, dt_cv_score = train_and_evaluate_classification_model(
    dt_base, param_grid_dt, X_train_scaled, y_train, cv=5
)

y_pred_dt = best_dt.predict(X_test_scaled)
y_prob_dt = best_dt.predict_proba(X_test_scaled)[:, 1]

dt_metrics = compute_classification_metrics(y_test, y_pred_dt, y_prob_dt)

print("--- Decision Tree Best Hyperparameters ---")
for k, v in dt_params.items():
    print(f"  {k}: {v}")
print(f"Best CV F1-Score: {dt_cv_score:.4f}")
print("\nTest Set Performance Metrics:")
for k, v in dt_metrics.items():
    print(f"  {k:10s}: {v:.4f}")


In [ ]:
# 2. Train and Tune Random Forest Classifier
rf_base = RandomForestClassifier(random_state=42)

param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True, False]
}

best_rf, rf_params, rf_cv_score = train_and_evaluate_classification_model(
    rf_base, param_grid_rf, X_train_scaled, y_train, cv=5
)

y_pred_rf = best_rf.predict(X_test_scaled)
y_prob_rf = best_rf.predict_proba(X_test_scaled)[:, 1]

rf_metrics = compute_classification_metrics(y_test, y_pred_rf, y_prob_rf)

print("--- Random Forest Best Hyperparameters ---")
for k, v in rf_params.items():
    print(f"  {k}: {v}")
print(f"Best CV F1-Score: {rf_cv_score:.4f}")
print("\nTest Set Performance Metrics:")
for k, v in rf_metrics.items():
    print(f"  {k:10s}: {v:.4f}")


In [ ]:
# Performance Comparison Summary Table
comparison_df = pd.DataFrame([dt_metrics, rf_metrics], index=['Decision Tree', 'Random Forest'])
print("=== Model Performance Comparison ===")
print(comparison_df.round(4))


In [ ]:
# Result Visualization 1: Side-by-Side Confusion Matrices (Optimized Compact EPS)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_dt = confusion_matrix(y_test, y_pred_dt)
cm_rf = confusion_matrix(y_test, y_pred_rf)

for ax, cm, title, cmap in zip(axes, [cm_dt, cm_rf], ['Decision Tree', 'Random Forest'], ['Blues', 'Greens']):
    ax.imshow(cm, cmap=cmap, aspect='equal', rasterized=True)
    ax.set_title(f'{title} Confusion Matrix')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Malignant (0)', 'Benign (1)'])
    ax.set_yticklabels(['Malignant (0)', 'Benign (1)'])
    ax.set_xlabel('Predicted Label', fontweight='bold')
    ax.set_ylabel('True Label', fontweight='bold')
    
    thresh = cm.max() / 2.
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black', fontsize=16, fontweight='bold')

plt.tight_layout()
fig.savefig('exp5_confusion_matrices.webp', format='webp', dpi=300, bbox_inches='tight')
fig.savefig('exp5_confusion_matrices.eps', format='eps', dpi=72, bbox_inches='tight')
plt.show()


In [ ]:
# Result Visualization 2: Combined ROC Curves
fpr_dt, tpr_dt, _ = roc_curve(y_test, 1 - y_prob_dt, pos_label=0)
fpr_rf, tpr_rf, _ = roc_curve(y_test, 1 - y_prob_rf, pos_label=0)

plt.figure(figsize=(8, 6))
plt.plot(fpr_dt, tpr_dt, label=f"Decision Tree (AUC = {dt_metrics['ROC-AUC']:.4f})", color='#d62728', linewidth=2)
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {rf_metrics['ROC-AUC']:.4f})", color='#2ca02c', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')

plt.xlabel('False Positive Rate', fontweight='bold')
plt.ylabel('True Positive Rate', fontweight='bold')
plt.title('ROC Curves Comparison')
plt.legend(loc='lower right')

plt.tight_layout()
plt.savefig('exp5_roc_curves.webp', format='webp', dpi=300, bbox_inches='tight')
plt.savefig('exp5_roc_curves.eps', format='eps', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# Result Visualization 3: Performance Metrics Comparison Bar Chart
metrics_names = list(dt_metrics.keys())
x = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width/2, [dt_metrics[m] for m in metrics_names], width, label='Decision Tree', color='#1f77b4', edgecolor='black')
ax.bar(x + width/2, [rf_metrics[m] for m in metrics_names], width, label='Random Forest', color='#2ca02c', edgecolor='black')

ax.set_ylabel('Score', fontweight='bold')
ax.set_title('Test Set Metrics Comparison')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names, fontweight='bold')
ax.set_ylim(0.85, 1.02)
ax.legend(loc='lower right')

plt.tight_layout()
fig.savefig('exp5_metrics_comparison.webp', format='webp', dpi=300, bbox_inches='tight')
fig.savefig('exp5_metrics_comparison.eps', format='eps', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# Result Visualization 4: Top 10 Feature Importances
importances_dt = best_dt.feature_importances_
importances_rf = best_rf.feature_importances_
top_idx = np.argsort(importances_rf)[::-1][:10]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(np.arange(10) - 0.2, importances_dt[top_idx], 0.4, label='Decision Tree', color='#d62728', edgecolor='black')
ax.barh(np.arange(10) + 0.2, importances_rf[top_idx], 0.4, label='Random Forest', color='#2ca02c', edgecolor='black')

ax.set_yticks(np.arange(10))
ax.set_yticklabels(X.columns[top_idx], fontweight='bold')
ax.set_xlabel('Gini Feature Importance', fontweight='bold')
ax.set_title('Top 10 Feature Importances')
ax.invert_yaxis()
ax.legend(loc='lower right')

plt.tight_layout()
fig.savefig('exp5_feature_importances.webp', format='webp', dpi=300, bbox_inches='tight')
fig.savefig('exp5_feature_importances.eps', format='eps', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# Result Visualization 5: Decision Tree Structure Visualization
plt.figure(figsize=(18, 10))
plot_tree(
    best_dt,
    feature_names=X.columns,
    class_names=['Malignant', 'Benign'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title('Tuned Decision Tree Structure', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.savefig('exp5_decision_tree_structure.webp', format='webp', dpi=300, bbox_inches='tight')
plt.savefig('exp5_decision_tree_structure.eps', format='eps', dpi=100, bbox_inches='tight')
plt.show()
